# Malware Detection Pipeline: folder2parquet → YARA

This notebook demonstrates the end-to-end workflow for scanning raw files for malware signatures using the DPK transform pipeline:

1. **folder2parquet** — Ingest raw files from a directory into parquet format (produces `binary_contents` column)
2. **yara** — Scan the binary content against YARA rules and annotate each row with match results
3. **Analysis** — Inspect results, separate clean from infected files

This mirrors a real-world use case: you have a folder of source files (code, scripts, binaries) and want to identify which ones contain known malware signatures before including them in an LLM training dataset.

## Setup

This notebook assumes you are running from the `transforms/code/yara/` directory with the venv activated:
```bash
make venv
source venv/bin/activate
uv pip install jupyterlab
venv/bin/jupyter lab
```

In [1]:
%%capture
# Reference only — install from pypi for non-developer usage:
# %uv pip install 'data-prep-toolkit-transforms[folder2parquet,yara]'
! uv pip install pandas

In [1]:
import os
import shutil
import pandas as pd

## Step 0: Create sample input files

We create a small directory of files to simulate a real ingestion scenario:
- `hello.py` — A benign Python script
- `utils.sh` — A benign shell script
- `malicious.txt` — Contains the EICAR anti-virus test string (industry-standard harmless test signature)
- `readme.txt` — A benign text file

In [3]:
# Clean up any previous run
for d in ["pipeline-demo/raw-files", "pipeline-demo/ingested", "pipeline-demo/scanned"]:
    shutil.rmtree(d, ignore_errors=True)

os.makedirs("pipeline-demo/raw-files", exist_ok=True)

# Benign files
with open("pipeline-demo/raw-files/hello.py", "w") as f:
    f.write('def greet(name):\n    return f"Hello, {name}!"\n\nif __name__ == "__main__":\n    print(greet("world"))\n')

with open("pipeline-demo/raw-files/utils.sh", "w") as f:
    f.write('#!/bin/bash\necho "Running cleanup..."\nfind /tmp -name "*.log" -mtime +7 -delete\n')

with open("pipeline-demo/raw-files/readme.txt", "w") as f:
    f.write("This project contains utility scripts for data processing.\n")

# File containing the EICAR test string (standard anti-virus test signature)
EICAR = r"X5O!P%@AP[4\PZX54(P^)7CC)7}$EICAR-STANDARD-ANTIVIRUS-TEST-FILE!$H+H*"
with open("pipeline-demo/raw-files/malicious.txt", "w") as f:
    f.write(EICAR)

print("Created sample files:")
for fname in sorted(os.listdir("pipeline-demo/raw-files")):
    size = os.path.getsize(f"pipeline-demo/raw-files/{fname}")
    print(f"  {fname} ({size} bytes)")

Created sample files:
  hello.py (100 bytes)
  malicious.txt (68 bytes)
  readme.txt (59 bytes)
  utils.sh (80 bytes)


## Step 1: Ingest files with folder2parquet

The `Folder2Parquet` transform reads every matching file and produces a parquet table with columns:
- `file_name` — Relative path of the source file
- `document_uuid` — Unique identifier for each document
- `binary_contents` — Raw file bytes

In [2]:
from dpk_folder2parquet.runtime import Folder2Parquet

Folder2Parquet(
    input_folder="pipeline-demo/raw-files",
    output_folder="pipeline-demo/ingested",
    data_files_to_use=[".py", ".sh", ".txt"],
).transform()

14:08:56 [INFO] transform.py:204 - Ingest file parameters are : {'f2p_content_column': 'binary_contents', 'f2p_fewer_parquets': False, 'f2p_file_name': 'file_name', 'f2p_document_uuid': 
'document_uuid'}
taskName=Task-47

14:08:56 [INFO] execution_configuration.py:91 - pipeline id pipeline_id
taskName=Task-47

14:08:56 [INFO] execution_configuration.py:94 - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'commit_hash': 'UNDEFINED', 'path': 'UNDEFINED'}
taskName=Task-47

14:08:56 [INFO] data_access_factory.py:257 - data factory data_ max_files -1, n_sample -1
taskName=Task-47

14:08:56 [INFO] data_access_factory.py:271 - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.py', '.sh', '.txt'], files to checkpoint 
['.parquet']
taskName=Task-47

...

14:08:56 [INFO] transform_orchestrator.py:197 - Completed 4 files (100.0%) in 0.0 min
taskName=Task-47

14:08:56 [INFO] transform_orchestrator.py:201 - Done processing 4 files, waiting for flush() completion.
taskName=Task-47

14:08:56 [INFO] transform_orchestrator.py:205 - done flushing in 0.001 sec
taskName=Task-47

14:08:56 [INFO] transform_launcher.py:65 - Completed execution in 0.0 min, execution result 0
taskName=Task-47

0

In [3]:
# Inspect the ingested parquet
import glob

ingested_files = glob.glob("pipeline-demo/ingested/*.parquet")
print(f"Ingested parquet files: {ingested_files}\n")

df_ingested = pd.concat([pd.read_parquet(f) for f in ingested_files], ignore_index=True)
print(f"Total rows ingested: {len(df_ingested)}")
print(f"Columns: {list(df_ingested.columns)}\n")

# Show file names and content sizes (not the raw bytes)
df_ingested[["file_name"]].assign(
    content_size=df_ingested["binary_contents"].apply(lambda b: len(b) if b else 0)
)

Ingested parquet files: ['pipeline-demo/ingested/utils.parquet']

Total rows ingested: 4
Columns: ['file_name', 'document_uuid', 'binary_contents']



,file_name,content_size
0,hello.py,100
1,malicious.txt,68
2,readme.txt,59
3,utils.sh,80


## Step 2: Scan with YARA rules

The `Yara` transform reads the `binary_contents` column produced by folder2parquet, scans each row against compiled YARA rules, and appends annotation columns:
- `yara_matched` — Boolean flag (True if any rule matched)
- `yara_rules` — List of matched rule names
- `yara_tags` — List of tags from matched rules
- `yara_categories` — Which rule source(s) contributed the match

We use the committed `test-rules/` directory (EICAR rule only) for a hermetic demo. For production, use `make fetch-rules` to download the full rule sets.

In [4]:
from dpk_yara.runtime import Yara

Yara(
    input_folder="pipeline-demo/ingested",
    output_folder="pipeline-demo/scanned",
    yara_input_column="binary_contents",
    yara_rules_dir="test-rules",
).transform()

14:09:05 [INFO] transform.py:266 - yara parameters are : {'yara_input_column': 'binary_contents', 'yara_rules_dir': 'test-rules', 'yara_category': None, 'yara_matched_column': 'yara_matched', 
'yara_rules_column': 'yara_rules', 'yara_tags_column': 'yara_tags', 'yara_categories_column': 'yara_categories', 'yara_fail_on_compile_error': False}
taskName=Task-53

14:09:05 [INFO] execution_configuration.py:91 - pipeline id pipeline_id
taskName=Task-53

14:09:05 [INFO] execution_configuration.py:94 - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'commit_hash': 'UNDEFINED', 'path': 'UNDEFINED'}
taskName=Task-53

14:09:05 [INFO] data_access_factory.py:257 - data factory data_ max_files -1, n_sample -1
taskName=Task-53

14:09:05 [INFO] data_access_factory.py:271 - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
taskName=Task-53

...

14:09:05 [INFO] transform_orchestrator.py:197 - Completed 1 files (100.0%) in 0.0 min
taskName=Task-53

14:09:05 [INFO] transform_orchestrator.py:201 - Done processing 1 files, waiting for flush() completion.
taskName=Task-53

14:09:05 [INFO] transform_orchestrator.py:205 - done flushing in 0.0 sec
taskName=Task-53

14:09:05 [INFO] transform_launcher.py:65 - Completed execution in 0.0 min, execution result 0
taskName=Task-53

0

## Step 3: Inspect results

Load the scanned output and examine which files were flagged.

In [5]:
scanned_files = glob.glob("pipeline-demo/scanned/*.parquet")
df_scanned = pd.concat([pd.read_parquet(f) for f in scanned_files], ignore_index=True)

print(f"Total files scanned: {len(df_scanned)}")
print(f"Columns: {list(df_scanned.columns)}\n")

# Display results (drop binary_contents for readability)
df_scanned[["file_name", "yara_matched", "yara_rules", "yara_tags", "yara_categories"]]

Total files scanned: 4
Columns: ['file_name', 'document_uuid', 'binary_contents', 'yara_matched', 'yara_rules', 'yara_tags', 'yara_categories']



,file_name,yara_matched,yara_rules,yara_tags,yara_categories
0,hello.py,False,[],[],[]
1,malicious.txt,True,[EICAR_Test_String],[],[eicar]
2,readme.txt,False,[],[],[]
3,utils.sh,False,[],[],[]


## Step 4: Separate clean from infected

In a real pipeline you would use the DPK `filter` transform to quarantine matched rows. Here we demonstrate the logic directly.

In [6]:
df_clean = df_scanned[df_scanned["yara_matched"] == False]
df_infected = df_scanned[df_scanned["yara_matched"] == True]

print(f"Clean files ({len(df_clean)}):")
for fname in df_clean["file_name"].tolist():
    print(f"  ✓ {fname}")

print(f"\nInfected files ({len(df_infected)}):")
for _, row in df_infected.iterrows():
    print(f"  ✗ {row['file_name']} — matched rules: {row['yara_rules']}")

Clean files (3):
  ✓ hello.py
  ✓ readme.txt
  ✓ utils.sh

Infected files (1):
  ✗ malicious.txt — matched rules: ['EICAR_Test_String']


## Summary

This pipeline demonstrates the core DPK malware detection workflow:

```
Raw files on disk
      │
      ▼
┌─────────────────┐
│ folder2parquet  │  Ingest → parquet with binary_contents
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   YARA scan     │  Annotate → yara_matched, yara_rules, ...
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ Filter/Quarantine│  Keep clean rows for LLM training
└─────────────────┘
```

For production use:
- Run `make fetch-rules` to download the full ReversingLabs + Trellix rule sets (~1600 rules)
- Use the Ray runtime for distributed scanning of large datasets
- Deploy via `granite.build/` configs for IBM Cloud scale